# Introduction to R in the Age of AI
### A 2-Hour Hands-On Workshop — Summer 2026

> **Workshop philosophy:** AI tools like ChatGPT and Claude can now write R code for you.
> That makes understanding R *more* important, not less — you need to read, verify, and debug
> what AI produces. This workshop teaches you to be a critical AI collaborator, not a passive user.

---

## Outline (120 minutes)

| # | Topic | Time |
|---|-------|------|
| 1 | Why R Still Matters | 10 min |
| 2 | Setup & Basic Syntax | 10 min |
| 3 | Data Structures That You'll Actually Use | 15 min |
| 4 | Data Wrangling with Real Data | 20 min |
| 5 | Visualization with ggplot2 | 15 min |
| 6 | Statistical Thinking in 10 Minutes | 10 min |
| 7 | Your First ML Workflow | 15 min |
| 8 | Working with AI: Prompting, Verifying, Fixing | 15 min |
| – | Wrap-up + Q&A | 10 min |

---

## ▶ Run This First

Install and load all packages needed for the workshop.

In [ ]:
#@title ▶ Setup — run this cell first { display-mode: "form" }
pkgs <- c("ggplot2", "dplyr", "tidyr", "rpart", "rpart.plot", "Metrics")
installed <- rownames(installed.packages())
to_install <- pkgs[!pkgs %in% installed]
if (length(to_install)) install.packages(to_install)

library(ggplot2)
library(dplyr)
library(tidyr)
library(rpart)
library(rpart.plot)
library(Metrics)
cat("✅ All packages loaded.\n")

---

# 1 · Why R Still Matters

**The AI question everyone is thinking:** *"If ChatGPT can write R code, why learn R?"*

Here's the honest answer:

| What AI is good at | What YOU still need |
|---|---|
| Writing boilerplate code | Understanding what the code does |
| Suggesting packages | Knowing whether the approach is valid |
| Fixing syntax errors | Catching logic errors and wrong assumptions |
| Generating plots | Interpreting what the plot means |
| Explaining concepts | Deciding which method fits your data |

AI is a *coding assistant*, not a *data scientist*. The data scientist is you.

### R's unique strengths in 2026
- **Statistical rigor** — reproducible, peer-reviewed methods built in  
- **Visualization** — ggplot2 grammar is unmatched for publication figures  
- **Bioinformatics / genomics** — Bioconductor has no Python equivalent  
- **RMarkdown / Quarto** — narrative + code + output in one document  
- **HPC integration** — submit parallel R jobs to SLURM just like any other workflow  

> **Key mindset shift:** R + AI together is more powerful than either alone.
> Your job is to direct the collaboration intelligently.

---

# 2 · Setup & Basic Syntax

## Assignment and naming

R uses `<-` by convention (you'll see both `<-` and `=` in the wild).

In [ ]:
# R convention: use <- for assignment
gene_count <- 24000
sample_id  <- "SRR123456"
is_paired  <- TRUE

# Print values
gene_count
sample_id
is_paired

## Arithmetic and comparisons

In [ ]:
# Basic arithmetic
2 + 2
3.14 * 10^2
sqrt(144)

# Comparisons return TRUE/FALSE — critical for filtering data
gene_count > 20000
sample_id == "SRR999"

## Getting help

```r
?mean           # help page for a function
??normalization  # search for keyword across all installed packages
```

> 💡 **AI tip:** If the help page is confusing, paste the function signature into Claude or ChatGPT and ask "explain this in plain English with an example."

---

# 3 · Data Structures That You'll Actually Use

We focus on the two you'll encounter 95% of the time:
**vectors** and **data frames**.

### Vectors

In [ ]:
# Numeric vector — e.g. expression values for 5 genes
expr <- c(12.4, 0.3, 88.1, 45.0, 7.7)
expr

# Character vector — gene names
genes <- c("BRCA1", "TP53", "MYC", "EGFR", "KRAS")
genes

# Logical vector — which genes are highly expressed (> 40)?
high_expr <- expr > 40
high_expr

# Use that logical vector to filter
genes[high_expr]   # <- This pattern is everywhere in R

### Data Frames — the workhorse of R

In [ ]:
# Build a small data frame from scratch
gene_df <- data.frame(
  gene    = genes,
  expr    = expr,
  is_high = high_expr,
  stringsAsFactors = FALSE
)

gene_df

# Basic inspection
nrow(gene_df)
ncol(gene_df)
str(gene_df)

### Subsetting — three ways to slice

Understanding these is *essential* for reading AI-generated code.

In [ ]:
# 1. By column name (most readable)
gene_df$expr

# 2. By position [row, col]
gene_df[1, ]      # first row
gene_df[, 2]      # second column

# 3. By condition
gene_df[gene_df$expr > 40, ]

> **⚠️ Watch out:** When AI generates subsetting code it often mixes these styles.
> If you see `df[df$x > 5, "y"]`, that's style 2 + 3 combined — read it as:
> *"rows where x > 5, only column y"*.

---

# 4 · Data Wrangling with Real Data

We'll use the built-in `mtcars` dataset (car performance specs) — small enough to understand
instantly, realistic enough to show real patterns.

We'll use **dplyr** — the tidyverse data wrangling package.
Its verbs (`filter`, `select`, `mutate`, `summarize`, `arrange`) map almost directly
to SQL, and AI tools know them very well.

In [ ]:
# Load and preview
data(mtcars)
head(mtcars)

In [ ]:
# --- dplyr verbs ---

# filter(): keep rows matching a condition
efficient <- filter(mtcars, mpg > 25)
efficient

# select(): keep only certain columns
select(mtcars, mpg, cyl, hp)

# mutate(): add or transform a column
mtcars <- mutate(mtcars, kpl = mpg * 0.425)  # miles/gallon → km/litre
head(mtcars[, c("mpg", "kpl")])

In [ ]:
# arrange(): sort rows
arrange(mtcars, desc(mpg))   # best fuel efficiency first

In [ ]:
# summarize() + group_by(): aggregate statistics by group
mtcars %>%
  group_by(cyl) %>%
  summarize(
    n         = n(),
    mean_mpg  = round(mean(mpg), 1),
    mean_hp   = round(mean(hp), 1),
    mean_wt   = round(mean(wt), 2)
  )

### The pipe `%>%` (or `|>`)

The pipe chains operations left-to-right — reads like a recipe.

```r
data %>%
  step1() %>%
  step2() %>%
  step3()
```

> 💡 **AI tip:** When you ask AI to wrangle data, it will almost always use pipes.
> Read them top-to-bottom: each step's output becomes the next step's input.

### 🔬 Quick exercise

Using the code above as a template, find the average horsepower (`hp`) and weight (`wt`)
for cars with 4 and 6 cylinders only (exclude `cyl == 8`).

In [ ]:
# Your code here
# Hint: chain filter() before group_by()

---

# 5 · Visualization with ggplot2

ggplot2 uses a **grammar of graphics**: every plot is built from layers.

```
ggplot(data, aes(x = ..., y = ..., color = ...))  # canvas + mapping
  + geom_*()                                        # geometric layer
  + labs()                                          # labels
  + theme_*()                                       # styling
```

Once you know this grammar, you can read any ggplot2 code AI generates.

In [ ]:
# Scatter plot: weight vs fuel efficiency, colored by cylinders
ggplot(mtcars, aes(x = wt, y = mpg, color = factor(cyl))) +
  geom_point(size = 3, alpha = 0.8) +
  geom_smooth(method = "lm", se = FALSE) +  # regression line per group
  labs(
    title = "Fuel Efficiency vs Weight",
    subtitle = "By number of cylinders",
    x = "Weight (1000 lbs)",
    y = "Miles per Gallon",
    color = "Cylinders"
  ) +
  theme_minimal()

In [ ]:
# Box plot: distribution of mpg by cylinder count
ggplot(mtcars, aes(x = factor(cyl), y = mpg, fill = factor(cyl))) +
  geom_boxplot(alpha = 0.7, outlier.shape = 21) +
  labs(title = "MPG Distribution by Cylinder Count",
       x = "Cylinders", y = "Miles per Gallon") +
  theme_bw() +
  theme(legend.position = "none")

In [ ]:
# Histogram with density overlay
ggplot(mtcars, aes(x = mpg)) +
  geom_histogram(aes(y = after_stat(density)), bins = 12,
                 fill = "#4E79A7", color = "white", alpha = 0.8) +
  geom_density(color = "#E15759", linewidth = 1.2) +
  labs(title = "Distribution of Fuel Efficiency",
       x = "Miles per Gallon", y = "Density") +
  theme_minimal()

> **AI + ggplot2 workflow:** 
> Describe your desired plot to AI in plain English.
> Paste the generated code here and run it.
> If it's wrong, copy the error message back to AI.
> Iterate until correct — then make sure you understand every line.

### 🔬 Quick exercise

Ask Claude or ChatGPT: *"Write ggplot2 R code for a bar chart showing average horsepower
per cylinder count in mtcars, with a clean minimal theme."*

Paste the code below and verify it runs correctly.

In [ ]:
# Paste AI-generated ggplot2 code here and run it

---

# 6 · Statistical Thinking in 10 Minutes

AI can run statistics for you. But you need to know *which test* to ask for
and whether the *result makes sense*.

## Correlation

In [ ]:
# How strongly are weight and fuel efficiency correlated?
cor(mtcars$wt, mtcars$mpg)
# r = -0.87 → strong negative correlation (heavier cars get worse mileage)

## Linear Regression

In [ ]:
# Simple linear regression: predict mpg from weight
model_lm <- lm(mpg ~ wt, data = mtcars)
summary(model_lm)

In [ ]:
# Multiple regression: add horsepower and cylinders
model_multi <- lm(mpg ~ wt + hp + cyl, data = mtcars)
summary(model_multi)

# Key things to read in summary():
# - Estimate: slope for each predictor
# - Pr(>|t|): p-value (< 0.05 is conventionally "significant")
# - R-squared: % of variance explained (0.84 = 84% here)
# - Residual standard error: typical prediction error

## t-test: comparing two groups

In [ ]:
# Do 4-cylinder and 8-cylinder cars differ significantly in MPG?
cars_4cyl <- filter(mtcars, cyl == 4)$mpg
cars_8cyl <- filter(mtcars, cyl == 8)$mpg

t.test(cars_4cyl, cars_8cyl)
# p < 0.05 → yes, the difference is statistically significant

> **Statistical responsibility in the AI era:**
> AI can generate a p-value. Only you can decide if the question being tested is sensible,
> whether the data meets the test's assumptions, and what the result actually means
> in your domain. Statistics requires judgment, not just computation.

---

# 7 · Your First ML Workflow

Machine learning in R follows a consistent pattern.
Understanding this pattern lets you evaluate AI-generated ML code critically.

**Data:** `iris` — classic classification dataset (150 flowers, 3 species, 4 measurements)

**Goal:** Predict flower species from petal and sepal measurements.

In [ ]:
data(iris)
str(iris)
table(iris$Species)   # 50 samples per species — balanced dataset

### Step 1: Visualize before modeling

In [ ]:
ggplot(iris, aes(x = Petal.Length, y = Petal.Width, color = Species)) +
  geom_point(size = 2.5, alpha = 0.8) +
  labs(title = "Iris: Petal Dimensions by Species",
       subtitle = "Well-separated clusters → classification should work well") +
  theme_minimal()

### Step 2: Train/test split

In [ ]:
set.seed(42)   # reproducibility — always set a seed!

train_idx  <- sample(1:nrow(iris), size = 0.75 * nrow(iris))
train_data <- iris[ train_idx, ]
test_data  <- iris[-train_idx, ]

cat("Training samples:", nrow(train_data), "\n")
cat("Test samples:    ", nrow(test_data), "\n")

### Step 3: Train a decision tree

In [ ]:
# Decision trees: human-readable, no black box
tree_model <- rpart(Species ~ ., data = train_data, method = "class")

# Visualize the tree — this is R's superpower for explainability
rpart.plot(tree_model,
           type = 4, extra = 104,
           box.palette = "RdYlGn",
           main = "Decision Tree: Iris Classification")

### Step 4: Predict and evaluate

In [ ]:
predictions <- predict(tree_model, test_data, type = "class")

# Confusion matrix — rows = actual, cols = predicted
conf_matrix <- table(Actual = test_data$Species, Predicted = predictions)
conf_matrix

# Overall accuracy
accuracy <- mean(predictions == test_data$Species)
cat(sprintf("\nAccuracy: %.1f%%\n", accuracy * 100))

### Step 5: Interpret the results

```
Looking at the confusion matrix:
- Perfect classification of setosa (100%)  
- A few versicolor/virginica confusions (these species overlap in petal size)
- Overall ~95% accuracy
```

> **Why this matters:** AI can generate all the code above in seconds.
> What AI *cannot* do is tell you whether 95% accuracy is good *for your use case*,
> whether your train/test split might have data leakage, or whether a decision tree
> is the right model for your domain constraints.

---

# 8 · Working with AI: Prompting, Verifying, Fixing

This is the section that makes everything else in this workshop more useful.

## The AI-R collaboration workflow

```
You describe goal → AI writes code → You run & read output
→ You verify logic → You ask AI to fix/explain → Repeat
```

## How to write good prompts for R code

### ❌ Vague prompt
> "Write R code to analyze my data"

### ✅ Specific prompt
> "I have an R data frame called `patient_df` with columns: `age` (numeric),
> `treatment` (character: 'A' or 'B'), and `outcome` (numeric, 0–100).
> Write dplyr code to compute the mean and standard deviation of `outcome`
> grouped by `treatment`, and then a ggplot2 boxplot comparing the two groups.
> Use theme_minimal()."

**The better your prompt describes your data structure and goal,
the more accurate the generated code will be.**

---

## Live demo: AI-assisted debugging

Run this intentionally broken code, then use AI to fix it.

In [ ]:
# ── BROKEN CODE — run this, read the error, then ask AI to fix it ──
# Paste the error message into Claude/ChatGPT with the code and ask:
# "This R code gives me the following error. What's wrong and how do I fix it?"

bad_df <- data.frame(x = 1:5, y = c(2, 4, NA, 8, 10))

bad_df %>%
  filter(x > 2) %>%
  sumarize(mean_y = mean(y))   # <-- intentional typo

In [ ]:
# ── FIXED VERSION (fill in after AI helps you) ──

bad_df <- data.frame(x = 1:5, y = c(2, 4, NA, 8, 10))

# Your corrected code here:

## Common AI code review checklist

Before using AI-generated R code, verify:

- [ ] **Does it run?** — Paste and execute; don't assume it works
- [ ] **Are the variable names right?** — AI sometimes invents column names
- [ ] **Is `na.rm = TRUE` needed?** — AI often forgets missing value handling
- [ ] **Is the package loaded?** — AI often uses functions without loading the library
- [ ] **Is the method appropriate?** — AI picks common methods, not necessarily the *right* one
- [ ] **Does the result match expectations?** — Sanity-check with `summary()` and a plot
- [ ] **Are you sharing private data?** — Never paste real patient/proprietary data into AI tools

---

## What AI is genuinely great at

| Task | Example prompt |
|---|---|
| Explaining code | "Explain this R function line by line" |
| Fixing errors | "Fix this error: `Error in ...: object not found`" |
| Finding packages | "What R package should I use for survival analysis?" |
| Code style | "Rewrite this for-loop using dplyr" |
| Plot polish | "Make this ggplot2 plot publication-ready with larger fonts" |
| Documentation | "Add comments to this R script explaining what each section does" |

---

## 🔬 Final exercise: Full AI-assisted workflow

**Task:** Using the World Happiness dataset, answer a real research question with AI assistance.

In [ ]:
# Download the data
url <- "https://raw.githubusercontent.com/ajaypalsinghlo/World-Happiness-Report/main/world-happiness-report.csv"
# (If URL is unavailable, use: data(mtcars) and adapt the exercise)

# Suggested workflow:
# 1. Load the data with read.csv()
# 2. Use str() and summary() to understand its structure
# 3. Ask AI: "Given this data structure: [paste str() output],
#             write dplyr + ggplot2 code to show how average 
#             happiness score changed over time by region"
# 4. Run the AI code, check for errors, fix with AI help
# 5. Interpret the result — what does it actually mean?

# Your code here:

---

# Wrap-Up: Key Takeaways

## What you can now do
1. **Read and write** R syntax, data structures, and dplyr/ggplot2 code
2. **Understand** AI-generated R code instead of copy-pasting blindly
3. **Debug** errors using both R skills and AI assistance
4. **Run** a complete ML workflow: load → explore → train → evaluate
5. **Ask** AI effective, specific prompts for R tasks

## Where to go next

| Goal | Resource |
|---|---|
| R fundamentals reference | [R for Data Science (free online)](https://r4ds.had.co.nz) |
| ggplot2 | [ggplot2 cheatsheet (Posit)](https://posit.co/resources/cheatsheets/) |
| Statistics in R | [Modern Statistics with R](https://www.modernstatisticswithr.com) |
| Machine learning in R | [Hands-On Machine Learning with R (free)](https://bradleyboehmke.github.io/HOML/) |
| HPC + R at LONI | [hpc.loni.org/training](https://hpc.loni.org/training) |
| Parallel R on SLURM | Ask your HPC support team — `doParallel` + `foreach` on multi-core nodes |

## Running R on HPC
Your analyses rarely stay small. When they grow:
- Submit R scripts as SLURM batch jobs
- Use `doParallel` / `foreach` for embarrassingly parallel tasks
- Use `Rscript` from the command line: `Rscript my_analysis.R`
- Load R via Lmod: `module load R`

---

*Workshop prepared for LONI/LSU HPC — Summer 2026*  
*Feedback: hpc@loni.org*